# LiteLLM 101

One small refund assistant, a primary model, and a backup. Call models through a common interface, test fallbacks, then start a shared proxy.


## Setup

In [ ]:
%pip install -q "litellm[proxy]==1.104.2" "openai==2.54.0" "httpx==0.28.1" "PyYAML==6.0.3"

### Key, models, and one question

Both models use the same provider to keep setup small. This demonstrates model failover; surviving a provider-wide outage requires a suitable backup on an independent provider.

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = (
    os.getenv("OPENAI_API_KEY") or getpass("OpenAI API key: ")
).strip()
if not os.environ["OPENAI_API_KEY"]:
    raise ValueError("An OpenAI API key is required.")

# Use the bundled model metadata; this lesson does not need a live price-map download.
os.environ["LITELLM_LOCAL_MODEL_COST_MAP"] = "True"

import litellm
from copy import deepcopy
from IPython.display import Markdown, display

PRIMARY_MODEL = "openai/gpt-4o-mini-2024-07-18"
BACKUP_MODEL = "openai/gpt-4.1-nano-2025-04-14"
MESSAGES = [
    {"role": "system", "content": "Our fictional policy allows refunds within 30 days of purchase. Answer using only this policy, in one short sentence."},
    {"role": "user", "content": "What is the refund window?"},
]

## 1. One interface for model calls

`completion()` accepts a provider/model name and messages. The response follows a common format. Changing the model does not require a new calling pattern, although credentials and supported features can differ.

In [ ]:
response = litellm.completion(
    model=PRIMARY_MODEL, messages=MESSAGES,
    temperature=0, max_tokens=80, timeout=45, num_retries=0,
)
print(response.choices[0].message.content)
print("Provider model:", response.model)

## 2. A primary model and a backup
### Routing rules

`primary` and `backup` are names our app uses. Each points to a provider model. A **retry** makes another attempt within the requested model group. A **fallback** moves to a configured backup group.

The rule below allows one retry, then a fallback from `primary` to `backup`. Each group has one deployment in this example.

In [ ]:
MODEL_LIST = [
    {
        "model_name": "primary",
        "litellm_params": {"model": PRIMARY_MODEL, "api_key": "os.environ/OPENAI_API_KEY"},
        "model_info": {"id": "primary-deployment"},
    },
    {
        "model_name": "backup",
        "litellm_params": {"model": BACKUP_MODEL, "api_key": "os.environ/OPENAI_API_KEY"},
        "model_info": {"id": "backup-deployment"},
    },
]
ROUTER_SETTINGS = {
    "fallbacks": [{"primary": ["backup"]}],
    "num_retries": 1,
    "retry_after": 0,
    "disable_cooldowns": True,  # Keep independent teaching tests repeatable.
}

router = litellm.Router(model_list=deepcopy(MODEL_LIST), **ROUTER_SETTINGS)

### A healthy request

The app requests `primary`. The deployment ID tells us which route actually served the response. It is stronger evidence than asking the model to identify itself.

In [ ]:
response = router.completion(
    model="primary", messages=MESSAGES, temperature=0, max_tokens=80, timeout=45,
)
served_by = response._hidden_params["model_id"]
print(response.choices[0].message.content)
print("Requested: primary | Served by:", served_by)
assert served_by == "primary-deployment", "Primary did not serve this request; check model access."

## 3. Fallback scenarios
### Controlled errors, real backup answers

The next cell injects an error into **only the primary deployment** of a fresh test Router. The backup still makes a real model call. The app keeps requesting `primary`.

| Scenario | What we simulate |
|---|---|
| Rate limit | The primary cannot accept another request. |
| Timeout | The primary does not respond within the allowed time. |
| Provider unavailable | The primary service cannot handle the request. |

`mock_response` is a test control. The timeout error is injected immediately, so we do not wait for an actual timeout. These tests exercise the Router's failure handling, not provider network behavior.

In [ ]:
SCENARIOS = [
    ("Rate limit", litellm.RateLimitError),
    ("Timeout", litellm.Timeout),
    ("Provider unavailable", litellm.ServiceUnavailableError),
]
results = []

for scenario, error_type in SCENARIOS:
    test_router = litellm.Router(model_list=deepcopy(MODEL_LIST), **ROUTER_SETTINGS)
    # Add the test exception after initialization; exception objects cannot be deep-copied.
    test_router.model_list[0]["litellm_params"]["mock_response"] = error_type(
        message=f"Simulated: {scenario}", model=PRIMARY_MODEL, llm_provider="openai",
    )
    response = test_router.completion(
        model="primary", messages=MESSAGES, temperature=0, max_tokens=80, timeout=45,
    )
    served_by = response._hidden_params["model_id"]
    assert served_by == "backup-deployment", f"Fallback was not verified for {scenario}."
    results.append((scenario, served_by))
    print(f"{scenario}: primary failed; served by {served_by}")
    print(response.choices[0].message.content, "\n")

display(Markdown(
    "| Scenario | Requested | Served by |\n|---|---|---|\n" +
    "\n".join(f"| {scenario} | primary | {deployment} |" for scenario, deployment in results)
))

**Insight:** our application requested the same name every time; the Router followed the backup rule. The deployment IDs verify the change. A fallback improves availability, but the backup can also fail or produce a different answer. It does not fix a quality problem in an otherwise successful response.

## 4. Starting the LiteLLM Proxy
### The same rules, behind a server

The SDK runs inside Python. The Proxy exposes a shared, OpenAI-compatible endpoint that other applications can call. It can also centralize keys, budgets, and usage tracking; this small demo needs no database or admin UI.

We reuse the clean model list. A third name, `unavailable-primary`, points to a deliberately unreachable local endpoint so we can test **real Proxy failover**. It falls back to the same `backup`.

In [ ]:
import secrets
import socket
import tempfile
from pathlib import Path
import yaml

# Reserve a local port without listening: connection attempts will fail.
if "unavailable_socket" in globals():
    unavailable_socket.close()
unavailable_socket = socket.socket()
unavailable_socket.bind(("127.0.0.1", 0))
unavailable_port = unavailable_socket.getsockname()[1]

proxy_models = deepcopy(MODEL_LIST)
proxy_models.append({
    "model_name": "unavailable-primary",
    "litellm_params": {
        "model": PRIMARY_MODEL, "api_key": "local-test-only",
        "api_base": f"http://127.0.0.1:{unavailable_port}/v1",
        "timeout": 2, "max_retries": 0,
    },
    "model_info": {"id": "unavailable-deployment"},
})
proxy_rules = deepcopy(ROUTER_SETTINGS)
proxy_rules["fallbacks"].append({"unavailable-primary": ["backup"]})
os.environ["LITELLM_MASTER_KEY"] = "sk-" + secrets.token_hex(24)
config = {
    "model_list": proxy_models,
    "router_settings": proxy_rules,
    "general_settings": {"master_key": "os.environ/LITELLM_MASTER_KEY"},
}
work_dir = Path(tempfile.mkdtemp(prefix="litellm-101-"))
config_path = work_dir / "config.yaml"
config_path.write_text(yaml.safe_dump(config, sort_keys=False))
print(config_path.read_text())  # Contains environment references, not the provider key.

### Spin-up and readiness

The next cell starts the Proxy in the background and checks its readiness. It listens only on this runtime's local address. In Colab, it is reachable from notebook code, not from your laptop's localhost.

The generated master key authorizes access to the Proxy. The separate OpenAI key authorizes its provider calls. Neither key appears in the saved configuration or output.

If startup fails, the cell prints the actual server log with the two demo keys redacted. The final log error identifies the cause; the generic process-exit message alone does not.

In [ ]:
import sys
import subprocess
import time
import httpx

# Stop our previous process if this cell is rerun.
if "proxy_process" in globals() and proxy_process.poll() is None:
    proxy_process.terminate()
    proxy_process.wait(timeout=10)
if "proxy_log" in globals():
    proxy_log.close()

with socket.socket() as port_socket:
    port_socket.bind(("127.0.0.1", 0))
    proxy_port = port_socket.getsockname()[1]
PROXY_URL = f"http://127.0.0.1:{proxy_port}"
proxy_log = (work_dir / "proxy.log").open("w")
os.environ["PYTHONIOENCODING"] = "utf-8"
os.environ["PYTHONUTF8"] = "1"
proxy_process = subprocess.Popen(
    [sys.executable, "-m", "litellm.proxy.proxy_cli", "--config", str(config_path),
     "--host", "127.0.0.1", "--port", str(proxy_port)],
    stdout=proxy_log, stderr=subprocess.STDOUT,
)

try:
    with httpx.Client(
        trust_env=False, timeout=2,
        headers={"Authorization": "Bearer " + os.environ["LITELLM_MASTER_KEY"]},
    ) as health_client:
        deadline = time.monotonic() + 60
        while time.monotonic() < deadline:
            if proxy_process.poll() is not None:
                raise RuntimeError(f"Proxy exited during startup (exit code {proxy_process.returncode}).")
            try:
                if health_client.get(PROXY_URL + "/health/readiness").status_code == 200:
                    break
            except httpx.RequestError:
                pass
            time.sleep(0.5)
        else:
            raise RuntimeError("Proxy did not become ready within 60 seconds.")
except Exception as exc:
    if proxy_process.poll() is None:
        proxy_process.terminate()
        try:
            proxy_process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proxy_process.kill()
            proxy_process.wait()
    proxy_log.close()
    details = (work_dir / "proxy.log").read_text(encoding="utf-8", errors="replace")[-10000:]
    for key_name in ("OPENAI_API_KEY", "LITELLM_MASTER_KEY"):
        secret = os.environ.get(key_name)
        if secret:
            details = details.replace(secret, "[REDACTED]")
    print("Python:", sys.version.split()[0], "| Platform:", sys.platform)
    print("Proxy startup log (last 10,000 characters):\n" + (details or "No log was written."))
    raise RuntimeError(f"{exc} See the startup log printed above for the cause.") from None

print("Proxy ready:", PROXY_URL)

### Calling it with the OpenAI client

The client now calls our local Proxy. It uses the public model name `primary`; LiteLLM chooses the configured provider model. The Proxy response header identifies the deployment that served the request.

In [ ]:
from openai import OpenAI

proxy_http = httpx.Client(trust_env=False)
proxy_client = OpenAI(
    base_url=PROXY_URL + "/v1", api_key=os.environ["LITELLM_MASTER_KEY"],
    timeout=45, max_retries=0, http_client=proxy_http,
)

def ask_proxy(model_name):
    raw = proxy_client.chat.completions.with_raw_response.create(
        model=model_name, messages=MESSAGES, temperature=0, max_tokens=80,
    )
    response = raw.parse()
    served_by = raw.headers.get("x-litellm-model-id")
    print(response.choices[0].message.content)
    print(f"Requested: {model_name} | Served by: {served_by}")
    return served_by

assert ask_proxy("primary") == "primary-deployment", "The primary route was not verified."

### A real connection failure through the Proxy

`unavailable-primary` cannot accept a connection. We send a normal request with no mock-testing flag. The server applies its fallback rule and the real backup model answers.

This controlled test proves connection-failure handling through the Proxy. It does not reproduce a real provider outage or claim that all network failures behave identically.

In [ ]:
assert ask_proxy("unavailable-primary") == "backup-deployment", "Proxy fallback was not verified."

## 5. Cleanup

Stop the local server and close the clients after the demonstration. The code removes the temporary configuration and log files it created.

In [ ]:
import shutil

proxy_client.close()
if proxy_process.poll() is None:
    proxy_process.terminate()
    try:
        proxy_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        proxy_process.kill()
        proxy_process.wait()
proxy_log.close()
unavailable_socket.close()
shutil.rmtree(work_dir)
os.environ.pop("LITELLM_MASTER_KEY", None)
print("Local proxy stopped. Temporary demo files removed.")